In [46]:
import csv
import pandas as pd

In [47]:
with open("cafa6/train/train_terms.tsv") as train_terms:
    terms = pd.read_csv(train_terms, delimiter="\t")
terms

,EntryID,term,aspect
0,Q5W0B1,GO:0000785,C
1,Q5W0B1,GO:0004842,F
2,Q5W0B1,GO:0051865,P
3,Q5W0B1,GO:0006275,P
4,Q5W0B1,GO:0006513,P
...,...,...,...
537022,Q06667,GO:0070481,P
537023,B1NF19,GO:0033075,P
537024,B1NF19,GO:0047052,F
537025,B1NF19,GO:0047056,F


In [48]:
terms_grouped = terms.groupby('EntryID').agg({
    'term': list,
    'aspect': list
})
terms_grouped

,term,aspect
EntryID,,
A0A023FBW4,[GO:0019958],[F]
A0A023FBW7,[GO:0019957],[F]
A0A023FDY8,[GO:0019957],[F]
A0A023FF81,[GO:0019958],[F]
A0A023FFB5,[GO:0019957],[F]
...,...,...
X2JI34,"[GO:0106223, GO:0051762]","[F, P]"
X4Y2L4,"[GO:0033906, GO:0030214]","[F, P]"
X5JA13,"[GO:0070062, GO:0005829, GO:0005515, GO:000014...","[C, C, F, C, C, C, C, P]"


In [49]:
with open("cafa6/train/train_taxonomy.tsv") as train_taxonomy:
    taxonomy = pd.read_csv(train_taxonomy, delimiter="\t")
taxonomy

,A0A0C5B5G6,9606
0,A0JNW5,9606
1,A0JP26,9606
2,A0PK11,9606
3,A1A4S6,9606
4,A1A519,9606
...,...,...
82398,Q9UTM1,284812
82399,Q9Y7I1,284812
82400,Q9Y7P7,284812
82401,Q9Y7Q3,284812


In [52]:
from Bio import SeqIO

data = []
for record in SeqIO.parse("cafa6/train/train_sequences.fasta", "fasta"):
    data.append({
        "EntryID": record.id.split("|")[1],
        "description": record.description,
        "sequence": str(record.seq), # Keep as str or list?
        "length": len(record.seq)
    })

# Convert to DataFrame
df = pd.DataFrame(data)
df = df.join(terms_grouped, on="EntryID")

df.head()

,EntryID,description,sequence,length,term,aspect
0,A0A0C5B5G6,sp|A0A0C5B5G6|MOTSC_HUMAN Mitochondrial-derive...,"(M, R, W, Q, E, M, G, Y, I, F, Y, P, R, K, L, R)",16,"[GO:0001649, GO:0033687, GO:0005615, GO:000563...","[P, P, C, C, C, P, P, P, P, P, P, F, P, F]"
1,A0JNW5,sp|A0JNW5|BLT3B_HUMAN Bridge-like lipid transf...,"(M, A, G, I, I, K, K, Q, I, L, K, H, L, S, R, ...",1464,"[GO:0120013, GO:0034498, GO:0005769, GO:012000...","[F, P, C, P, F, C, F, F]"
2,A0JP26,sp|A0JP26|POTB3_HUMAN POTE ankyrin domain fami...,"(M, V, A, E, V, C, S, M, P, A, A, S, A, V, K, ...",581,[GO:0005515],[F]
3,A0PK11,sp|A0PK11|CLRN2_HUMAN Clarin-2 OS=Homo sapiens...,"(M, P, G, W, F, K, K, A, W, Y, G, L, A, S, L, ...",232,"[GO:0007605, GO:0005515]","[P, F]"
4,A1A4S6,sp|A1A4S6|RHG10_HUMAN Rho GTPase-activating pr...,"(M, G, L, Q, P, L, E, F, S, D, C, Y, L, D, S, ...",786,"[GO:0005829, GO:0010008, GO:0005515, GO:000509...","[C, C, F, F, P]"


In [51]:
df['term'].isna().sum()

np.int64(0)

Clearly, all proteins in train set have labels